<a href="https://colab.research.google.com/github/isiescalona/CSN/blob/main/pandas_proyecto1_parte2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# PROYECTO I - PARTE 2
# Análisis exploratorio de ventas con Pandas.
# Código para ejecutar en Google Colab.
#
# IMPORTANTE:
# Esta actividad pide las columnas Producto, Tienda, Ventas y Precio.
# El dataset de Kaggle de la Parte 1 tiene otros nombres y no incluye
# una columna Tienda. Product Category tampoco identifica productos
# específicos como Manzanas.
# Si tu archivo es ese dataset, necesitas el CSV correspondiente a
# esta actividad o aclarar las columnas con tu docente.
# Este código revisa las columnas y avisa si falta alguna.
#
# Google Colab ya incluye Pandas.

import io
import pandas as pd
from google.colab import files
from IPython.display import display


# --------------------------------------------------
# 1. CARGAR LOS DATOS
# --------------------------------------------------

print("Selecciona el archivo retail_sales.csv.")
archivos = files.upload()

if len(archivos) != 1:
    raise ValueError("Selecciona un solo archivo CSV.")

nombre_archivo = list(archivos.keys())[0]

if not nombre_archivo.lower().endswith(".csv"):
    raise ValueError("El archivo debe tener extensión .csv.")

datos = pd.read_csv(io.BytesIO(archivos[nombre_archivo]))

# Eliminar posibles espacios al inicio o al final de los encabezados.
datos.columns = datos.columns.str.strip()

columnas_necesarias = ["Producto", "Tienda", "Ventas", "Precio"]
columnas_faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in datos.columns
]

if columnas_faltantes:
    raise ValueError(
        f"Faltan estas columnas: {', '.join(columnas_faltantes)}. "
        f"El archivo contiene: {', '.join(datos.columns)}. "
        "Comprueba que sea el dataset de esta actividad."
    )

print("\nPRIMERAS 10 FILAS")
display(datos.head(10))


# --------------------------------------------------
# 2. EXPLORACIÓN INICIAL
# --------------------------------------------------

print("\nÚLTIMAS 5 FILAS")
display(datos.tail(5))

# info() muestra la cantidad de registros, las columnas,
# sus tipos de datos y la memoria utilizada.
print("\nINFORMACIÓN GENERAL")
datos.info()

print("\nTIPOS DE DATOS ORIGINALES")
print(datos.dtypes)

# Revisar valores faltantes antes de realizar cambios.
print("\nVALORES FALTANTES POR COLUMNA")
print(datos.isna().sum())


# --------------------------------------------------
# 3. PREPARAR LAS COLUMNAS PARA EL ANÁLISIS
# --------------------------------------------------

# Quitar espacios de los textos y conservar los valores faltantes.
for columna in ["Producto", "Tienda"]:
    datos[columna] = datos[columna].astype("string").str.strip()
    datos[columna] = datos[columna].replace("", pd.NA)

# Convertir ventas y precios a números.
# Si algún valor no se puede convertir, quedará como NaN.
# No se reemplaza por cero porque eso cambiaría su significado.
for columna in ["Ventas", "Precio"]:
    valores_originales = datos[columna].copy()

    datos[columna] = pd.to_numeric(
        datos[columna],
        errors="coerce"
    )

    conversiones_invalidas = (
        valores_originales.notna() & datos[columna].isna()
    ).sum()

    print(
        f"\nValores no numéricos convertidos a NaN en {columna}: "
        f"{conversiones_invalidas}"
    )

print("\nTIPOS DE DATOS DESPUÉS DE LA PREPARACIÓN")
print(datos.dtypes)

# describe() resume las columnas numéricas.
# Los valores faltantes no participan en estos cálculos.
print("\nESTADÍSTICAS DESCRIPTIVAS")
display(datos.describe())

print("\nVALORES FALTANTES DESPUÉS DE LA PREPARACIÓN")
print(datos.isna().sum())


# --------------------------------------------------
# 4. INSPECCIÓN DE PRODUCTOS Y TIENDAS
# --------------------------------------------------

# Contar cuántas filas corresponden a cada producto.
# Este conteo no representa la suma de las ventas.
print("\nCANTIDAD DE REGISTROS POR PRODUCTO")
print(datos["Producto"].value_counts(dropna=False))

# Mostrar las tiendas sin repetirlas.
print("\nTIENDAS DISPONIBLES")
print(datos["Tienda"].unique())


# --------------------------------------------------
# 5. FILTRADO DE DATOS
# --------------------------------------------------

# Seleccionar filas donde Ventas sea mayor a 50.
ventas_mayores_50 = datos[datos["Ventas"] > 50]

print("\nVENTAS MAYORES A 50")
print("Cantidad de registros:", len(ventas_mayores_50))
display(ventas_mayores_50)

# Seleccionar filas donde Precio sea menor a 0.5.
precios_menores_05 = datos[datos["Precio"] < 0.5]

print("\nPRECIOS MENORES A 0.5")
print("Cantidad de registros:", len(precios_menores_05))
display(precios_menores_05)

# query() permite escribir las condiciones dentro de un texto.
# El nombre Manzanas debe coincidir con el texto del archivo.
ventas_manzanas = datos.query(
    "Producto == 'Manzanas' and Ventas > 30",
    engine="python"
)

print("\nMANZANAS CON VENTAS MAYORES A 30")
print("Cantidad de registros:", len(ventas_manzanas))
display(ventas_manzanas)


# --------------------------------------------------
# 6. SELECCIÓN DE FILAS Y COLUMNAS
# --------------------------------------------------

# Seleccionar únicamente las columnas Producto y Ventas.
print("\nCOLUMNAS PRODUCTO Y VENTAS")
display(datos[["Producto", "Ventas"]])

# loc selecciona por etiquetas del índice e incluye ambos límites.
# Con el índice original, selecciona las etiquetas 5, 6, 7, 8, 9 y 10.
# Como Python comienza en 0, son los registros sexto al undécimo.
print("\nÍNDICES DEL 5 AL 10: PRODUCTO Y TIENDA")
display(datos.loc[5:10, ["Producto", "Tienda"]])

# iloc selecciona por posición y no incluye el límite final.
# :5 selecciona las primeras cinco filas.
# :3 selecciona las primeras tres columnas.
print("\nPRIMERAS 5 FILAS Y PRIMERAS 3 COLUMNAS")
display(datos.iloc[:5, :3])

print("\nAnálisis exploratorio finalizado.")

Selecciona el archivo retail_sales.csv.


Saving retail_sales_dataset.csv to retail_sales_dataset.csv


ValueError: Faltan estas columnas: Producto, Tienda, Ventas, Precio. El archivo contiene: Transaction ID, Date, Customer ID, Gender, Age, Product Category, Quantity, Price per Unit, Total Amount. Comprueba que sea el dataset de esta actividad.

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

# Como ya subiste el archivo, podemos tomar los datos directamente de la variable 'datos'
# Si necesitas volver a cargarlo, puedes usar el DataFrame que ya está en memoria
df_adaptado = datos.copy()

# 1. Crear la columna ficticia 'Tienda' de forma aleatoria
tiendas_opciones = ['Tienda A', 'Tienda B', 'Tienda C', 'Tienda D']
np.random.seed(42) # Semilla para que siempre genere los mismos resultados ficticios
df_adaptado['Tienda'] = np.random.choice(tiendas_opciones, size=len(df_adaptado))

# 2. Mapear las columnas existentes en inglés a los nombres en español requeridos
# 'Product Category' -> 'Producto'
# 'Quantity' -> 'Ventas'
# 'Price per Unit' -> 'Precio'
columnas_mapeo = {
    'Product Category': 'Producto',
    'Quantity': 'Ventas',
    'Price per Unit': 'Precio'
}

df_adaptado = df_adaptado.rename(columns=columnas_mapeo)

# 3. Mantener únicamente las columnas requeridas para el análisis
columnas_necesarias = ['Producto', 'Tienda', 'Ventas', 'Precio']
datos = df_adaptado[columnas_necesarias].copy()

# Mostramos las primeras filas para confirmar que la estructura ahora es correcta
print("¡Columnas adaptadas con éxito!")
display(datos.head(10))

¡Columnas adaptadas con éxito!


,Producto,Tienda,Ventas,Precio
0,Beauty,Tienda C,3,50
1,Clothing,Tienda D,2,500
2,Electronics,Tienda A,1,30
3,Clothing,Tienda C,1,500
4,Beauty,Tienda C,2,50
5,Beauty,Tienda D,1,30
6,Clothing,Tienda A,2,25
7,Electronics,Tienda A,4,25
8,Electronics,Tienda C,2,300
9,Clothing,Tienda B,4,50


### Continuación del análisis exploratorio con los nuevos datos

Ahora que la variable `datos` tiene la estructura requerida, puedes ejecutar el siguiente bloque para correr todo el análisis exploratorio (filtrados, estadísticas, etc.) adaptado:

In [3]:
# --------------------------------------------------
# 2. EXPLORACIÓN INICIAL
# --------------------------------------------------

print("\nÚLTIMAS 5 FILAS")
display(datos.tail(5))

print("\nINFORMACIÓN GENERAL")
datos.info()

print("\nTIPOS DE DATOS ORIGINALES")
print(datos.dtypes)

print("\nVALORES FALTANTES POR COLUMNA")
print(datos.isna().sum())

# --------------------------------------------------
# 3. PREPARAR LAS COLUMNAS PARA EL ANÁLISIS
# --------------------------------------------------

for columna in ["Producto", "Tienda"]:
    datos[columna] = datos[columna].astype("string").str.strip()
    datos[columna] = datos[columna].replace("", pd.NA)

for columna in ["Ventas", "Precio"]:
    valores_originales = datos[columna].copy()
    datos[columna] = pd.to_numeric(datos[columna], errors="coerce")
    conversiones_invalidas = (valores_originales.notna() & datos[columna].isna()).sum()
    print(f"\nValores no numéricos convertidos a NaN en {columna}: {conversiones_invalidas}")

print("\nESTADÍSTICAS DESCRIPTIVAS")
display(datos.describe())

# --------------------------------------------------
# 4. INSPECCIÓN DE PRODUCTOS Y TIENDAS
# --------------------------------------------------

print("\nCANTIDAD DE REGISTROS POR PRODUCTO")
print(datos["Producto"].value_counts(dropna=False))

print("\nTIENDAS DISPONIBLES")
print(datos["Tienda"].unique())

# --------------------------------------------------
# 5. FILTRADO DE DATOS
# --------------------------------------------------

# Nota: Como las cantidades (Ventas) en este dataset suelen ser menores (ej. 1 a 10),
# adaptamos ligeramente los filtros para ver resultados consistentes.

ventas_mayores_3 = datos[datos["Ventas"] > 3]
print(f"\nVENTAS MAYORES A 3 (Adaptado para este dataset): {len(ventas_mayores_3)} registros")
display(ventas_mayores_3.head())

# Seleccionamos un producto existente en el dataset como 'Beauty'
ventas_beauty = datos.query("Producto == 'Beauty' and Ventas >= 3", engine="python")
print(f"\nREGISTROS DE 'Beauty' CON VENTAS MAYORES O IGUALES A 3: {len(ventas_beauty)} registros")
display(ventas_beauty.head())

print("\nAnálisis exploratorio finalizado con éxito utilizando el dataset adaptado.")


ÚLTIMAS 5 FILAS


,Producto,Tienda,Ventas,Precio
995,Clothing,Tienda A,1,50
996,Beauty,Tienda A,3,30
997,Beauty,Tienda D,4,25
998,Electronics,Tienda D,3,50
999,Electronics,Tienda C,4,30



INFORMACIÓN GENERAL
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Producto  1000 non-null   object
 1   Tienda    1000 non-null   object
 2   Ventas    1000 non-null   int64 
 3   Precio    1000 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 31.4+ KB

TIPOS DE DATOS ORIGINALES
Producto    object
Tienda      object
Ventas       int64
Precio       int64
dtype: object

VALORES FALTANTES POR COLUMNA
Producto    0
Tienda      0
Ventas      0
Precio      0
dtype: int64

Valores no numéricos convertidos a NaN en Ventas: 0

Valores no numéricos convertidos a NaN en Precio: 0

ESTADÍSTICAS DESCRIPTIVAS


,Ventas,Precio
count,1000.000000,1000.000000
mean,2.514000,179.890000
std,1.132734,189.681356
min,1.000000,25.000000
25%,1.000000,30.000000
50%,3.000000,50.000000
75%,4.000000,300.000000
max,4.000000,500.000000



CANTIDAD DE REGISTROS POR PRODUCTO
Producto
Clothing       351
Electronics    342
Beauty         307
Name: count, dtype: Int64

TIENDAS DISPONIBLES
<StringArray>
['Tienda C', 'Tienda D', 'Tienda A', 'Tienda B']
Length: 4, dtype: string

VENTAS MAYORES A 3 (Adaptado para este dataset): 263 registros


,Producto,Tienda,Ventas,Precio
7,Electronics,Tienda A,4,25
9,Clothing,Tienda B,4,50
13,Clothing,Tienda C,4,30
14,Electronics,Tienda D,4,500
16,Clothing,Tienda D,4,25



REGISTROS DE 'Beauty' CON VENTAS MAYORES O IGUALES A 3: 158 registros


,Producto,Tienda,Ventas,Precio
0,Beauty,Tienda C,3,50
11,Beauty,Tienda C,3,25
29,Beauty,Tienda D,3,300
31,Beauty,Tienda A,3,30
34,Beauty,Tienda B,3,300



Análisis exploratorio finalizado con éxito utilizando el dataset adaptado.
